In [1]:
import torchvision.datasets as datasets

import os
import torch
import torch.nn as nn
import torchvision.transforms.v2
from torch.utils.data.dataset import Dataset
import xml.etree.ElementTree as ET
from torchvision import tv_tensors
from torchvision.io import read_image
import yaml
from tqdm import tqdm
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import random_split
from torch.optim.lr_scheduler import MultiStepLR
import torchvision.models
from torchvision.models import resnet34
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
import random

In [2]:
print(torch.cuda.is_available())

print(os.environ.get('CUDA_VISIBLE_DEVICES'))

print(torch.__version__)

print(torch.version.cuda)

True
0
2.1.2+cu118
11.8


In [3]:
# voc_detect_2012 = datasets.VOCDetection(
#     root='/scratch/abhilashk/Pascal-VOC', 
#     year='2012', 
#     image_set='trainval', 
#     download=False
# )

# voc_detect_2007 = datasets.VOCDetection(
#     root='/scratch/abhilashk/Pascal-VOC', 
#     year='2007', 
#     image_set='trainval', 
#     download=False
# )

In [4]:
# voc_detect_2012

In [5]:
# voc_detect_2007

In [6]:
def load_images_and_anns(im_sets, label2idx, ann_fname, split):
    r"""
    Method to get the xml files and for each file
    get all the objects and their ground truth detection
    information for the dataset
    :param im_sets: Sets of images to consider
    :param label2idx: Class Name to index mapping for dataset
    :param ann_fname: txt file containing image names{trainval.txt/test.txt}
    :param split: train/test
    :return:
    """
    im_infos = []
    for im_set in im_sets:
        im_names = []
        # Fetch all image names in txt file for this imageset
        for line in open(os.path.join(
                im_set, 'ImageSets', 'Main', '{}.txt'.format(ann_fname))):
            im_names.append(line.strip())

        # Set annotation and image path
        ann_dir = os.path.join(im_set, 'Annotations')
        im_dir = os.path.join(im_set, 'JPEGImages')

        for im_name in im_names:
            ann_file = os.path.join(ann_dir, '{}.xml'.format(im_name))
            im_info = {}
            ann_info = ET.parse(ann_file)
            root = ann_info.getroot()
            size = root.find('size')
            width = int(size.find('width').text)
            height = int(size.find('height').text)
            im_info['img_id'] = os.path.basename(ann_file).split('.xml')[0]
            im_info['filename'] = os.path.join(
                im_dir, '{}.jpg'.format(im_info['img_id'])
            )
            im_info['width'] = width
            im_info['height'] = height
            detections = []
            for obj in ann_info.findall('object'):
                det = {}
                label = label2idx[obj.find('name').text]
                difficult = int(obj.find('difficult').text)
                bbox_info = obj.find('bndbox')
                bbox = [
                    int(bbox_info.find('xmin').text) - 1,
                    int(bbox_info.find('ymin').text) - 1,
                    int(bbox_info.find('xmax').text) - 1,
                    int(bbox_info.find('ymax').text) - 1
                ]
                det['label'] = label
                det['bbox'] = bbox
                det['difficult'] = difficult
                detections.append(det)
            im_info['detections'] = detections
            # Because we are using 25 as num_queries,
            # so we ignore all images in VOC with greater
            # than 25 target objects.
            # This is okay, since this just means we are
            # ignoring a small number of images(15 to be precise)
            if len(detections) <= 25:
                im_infos.append(im_info)
    print('Total {} images found'.format(len(im_infos)))
    return im_infos

In [7]:
class VOCDataset(Dataset):
    def __init__(self, split, im_sets, im_size=640):
        self.split = split

        # Imagesets for this dataset instance (VOC2007/VOC2007+VOC2012/VOC2007-test)
        self.im_sets = im_sets
        self.fname = 'trainval' if self.split == 'train' else 'test'
        self.im_size = im_size
        self.im_mean = [123.0, 117.0, 104.0]
        self.imagenet_mean = [0.485, 0.456, 0.406]
        self.imagenet_std = [0.229, 0.224, 0.225]

        # Train and test transformations
        self.transforms = {
            'train': torchvision.transforms.v2.Compose([
                torchvision.transforms.v2.RandomHorizontalFlip(p=0.5),
                torchvision.transforms.v2.RandomZoomOut(fill=self.im_mean),
                torchvision.transforms.v2.RandomIoUCrop(),
                torchvision.transforms.v2.RandomPhotometricDistort(),
                torchvision.transforms.v2.Resize(size=(self.im_size, self.im_size)),
                torchvision.transforms.v2.SanitizeBoundingBoxes(
                    labels_getter=lambda transform_input:
                    transform_input[1]["labels"]),
                # torchvision.transforms.v2.SanitizeBoundingBoxes(
                #     labels_getter=lambda transform_input:
                #     (transform_input[1]["labels"], transform_input[1]["difficult"])),
                torchvision.transforms.v2.ToPureTensor(),
                torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
                torchvision.transforms.v2.Normalize(mean=self.imagenet_mean,
                                                    std=self.imagenet_std)

            ]),
            'test': torchvision.transforms.v2.Compose([
                torchvision.transforms.v2.Resize(size=(self.im_size, self.im_size)),
                torchvision.transforms.v2.ToPureTensor(),
                torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
                torchvision.transforms.v2.Normalize(mean=self.imagenet_mean,
                                                    std=self.imagenet_std)
            ]),
        }

        classes = [
            'person', 'bird', 'cat', 'cow', 'dog', 'horse', 'sheep',
            'aeroplane', 'bicycle', 'boat', 'bus', 'car', 'motorbike', 'train',
            'bottle', 'chair', 'diningtable', 'pottedplant', 'sofa', 'tvmonitor'
        ]
        classes = sorted(classes)
        # We need to add background class as well with 0 index
        classes = ['background'] + classes

        self.label2idx = {classes[idx]: idx for idx in range(len(classes))}
        self.idx2label = {idx: classes[idx] for idx in range(len(classes))}
        print(self.idx2label)
        self.images_info = load_images_and_anns(self.im_sets,
                                                self.label2idx,
                                                self.fname,
                                                self.split)

    def __len__(self):
        return len(self.images_info)

    def __getitem__(self, index):
        im_info = self.images_info[index]
        im = read_image(im_info['filename'])

        # Get annotations for this image
        targets = {}
        targets['boxes'] = tv_tensors.BoundingBoxes(
            [detection['bbox'] for detection in im_info['detections']],
            format='XYXY', canvas_size=im.shape[-2:])
        targets['labels'] = torch.as_tensor(
            [detection['label'] for detection in im_info['detections']])
        targets['difficult'] = torch.as_tensor(
            [detection['difficult']for detection in im_info['detections']])

        # Transform the image and targets
        transformed_info = self.transforms[self.split](im, targets)
        im_tensor, targets = transformed_info

        h, w = im_tensor.shape[-2:]

        # Boxes returned are in x1y1x2y2 format normalized from 0-1
        wh_tensor = torch.as_tensor([[w, h, w, h]]).expand_as(targets['boxes'])
        targets['boxes'] = targets['boxes'] / wh_tensor
        return im_tensor, targets, im_info['filename']

In [8]:
print(os.getcwd())

/home/abhilashk/MoR_Code


In [ ]:
# these files contain all hyperparameters for the models

config_path = '/home/abhilashk/MoR_Code/voc.yaml'
config_path_mor = '/home/abhilashk/MoR_Code/mor_voc.yaml'

In [11]:
with open(config_path_mor, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [12]:
def collate_function(data):
    return tuple(zip(*data))

dataset_config = config['dataset_params']
train_config = config['train_params']
model_config = config['model_params']

In [13]:
voc = VOCDataset('train',
                 im_sets=dataset_config['train_im_sets'],
                 im_size=dataset_config['im_size'])

# 2. Define split sizes (e.g., 80% train, 20% val)
train_size = int(0.8 * len(voc))
val_size = len(voc) - train_size

train, val = random_split(voc, [train_size, val_size])

train_dataset = DataLoader(train,
                           batch_size=train_config['batch_size'],
                           shuffle=True,
                           collate_fn=collate_function)

val_dataset = DataLoader(val,
                           batch_size=train_config['batch_size'],
                           shuffle=True,
                           collate_fn=collate_function)


{0: 'background', 1: 'aeroplane', 2: 'bicycle', 3: 'bird', 4: 'boat', 5: 'bottle', 6: 'bus', 7: 'car', 8: 'cat', 9: 'chair', 10: 'cow', 11: 'diningtable', 12: 'dog', 13: 'horse', 14: 'motorbike', 15: 'person', 16: 'pottedplant', 17: 'sheep', 18: 'sofa', 19: 'train', 20: 'tvmonitor'}
Total 16536 images found


In [14]:
train_dataset

In [ ]:
# Positional encoding for the vector embeddings for each image patch

def get_spatial_position_embedding(pos_emb_dim, feat_map):
    assert pos_emb_dim % 4 == 0, ('Position embedding dimension '
                                  'must be divisible by 4')
    grid_size_h, grid_size_w = feat_map.shape[2], feat_map.shape[3]
    grid_h = torch.arange(grid_size_h,
                          dtype=torch.float32,
                          device=feat_map.device)
    grid_w = torch.arange(grid_size_w,
                          dtype=torch.float32,
                          device=feat_map.device)
    grid = torch.meshgrid(grid_h, grid_w, indexing='ij')
    grid = torch.stack(grid, dim=0)

    # grid_h_positions -> (Number of grid cell tokens,)
    grid_h_positions = grid[0].reshape(-1)
    grid_w_positions = grid[1].reshape(-1)

    # factor = 10000^(2i/d_model)
    factor = 10000 ** ((torch.arange(
        start=0,
        end=pos_emb_dim // 4,
        dtype=torch.float32,
        device=feat_map.device) / (pos_emb_dim // 4))
    )

    grid_h_emb = grid_h_positions[:, None].repeat(1, pos_emb_dim // 4) / factor
    grid_h_emb = torch.cat([
        torch.sin(grid_h_emb),
        torch.cos(grid_h_emb)
    ], dim=-1)
    # grid_h_emb -> (Number of grid cell tokens, pos_emb_dim // 2)

    grid_w_emb = grid_w_positions[:, None].repeat(1, pos_emb_dim // 4) / factor
    grid_w_emb = torch.cat([
        torch.sin(grid_w_emb),
        torch.cos(grid_w_emb)
    ], dim=-1)
    pos_emb = torch.cat([grid_h_emb, grid_w_emb], dim=-1)

    # pos_emb -> (Number of grid cell tokens, pos_emb_dim)
    return pos_emb

# Base Model

In [16]:
class TransformerEncoder(nn.Module):
    r"""
    Encoder for transformer of DETR.
    This has sequence of encoder layers.
    Each layer has the following modules:
        1. LayerNorm for Self Attention
        2. Self Attention
        3. LayerNorm for MLP
        4. MLP
    """
    def __init__(self, num_layers, num_heads, d_model, ff_inner_dim,
                 dropout_prob=0.0):
        super().__init__()
        self.num_layers = num_layers
        self.dropout_prob = dropout_prob

        # Self Attention Module for all encoder layers
        self.attns = nn.ModuleList(
                [
                    nn.MultiheadAttention(d_model, num_heads,
                                          dropout=self.dropout_prob,
                                          batch_first=True)
                    for _ in range(num_layers)
                ]
            )

        # MLP Module for all encoder layers
        self.ffs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, ff_inner_dim),
                    nn.ReLU(),
                    nn.Dropout(self.dropout_prob),
                    nn.Linear(ff_inner_dim, d_model),
                )
                for _ in range(num_layers)
            ])

        # Norm for Self Attention for all encoder layers
        self.attn_norms = nn.ModuleList(
                [
                    nn.LayerNorm(d_model)
                    for _ in range(num_layers)
                ])

        # Norm for MLP for all encoder layers
        self.ff_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Dropout for Self Attention for all encoder layers
        self.attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Dropout for MLP for all encoder layers
        self.ff_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Norm for encoder output
        self.output_norm = nn.LayerNorm(d_model)

    def forward(self, x, spatial_position_embedding):
        out = x
        attn_weights = []
        for i in range(self.num_layers):
            # Norm, Self Attention, Dropout and Residual
            in_attn = self.attn_norms[i](out)
            # Add spatial position embedding
            # to q,k for self attention
            q = in_attn + spatial_position_embedding
            k = in_attn + spatial_position_embedding
            out_attn, attn_weight = self.attns[i](
                query=q,
                key=k,
                value=in_attn
            )
            attn_weights.append(attn_weight)
            out_attn = self.attn_dropouts[i](out_attn)
            out = out + out_attn

            # Norm, MLP, Dropout and Residual
            in_ff = self.ff_norms[i](out)
            out_ff = self.ffs[i](in_ff)
            out_ff = self.ff_dropouts[i](out_ff)
            out = out + out_ff

        # Output Normalization
        out = self.output_norm(out)
        return out, torch.stack(attn_weights)


In [17]:
class TransformerDecoder(nn.Module):
    r"""
        Decoder for transformer of DETR.
        This has sequence of decoder layers.
        Each layer has the following modules:
            1. LayerNorm for Self Attention
            2. Self Attention
            3. LayerNorm for Cross Attention on
                Encoder Outputs
            4. Cross Attention
            3. LayerNorm for MLP
            4. MLP
    """
    def __init__(self, num_layers, num_heads, d_model, ff_inner_dim,
                 dropout_prob=0.0):
        super().__init__()
        self.num_layers = num_layers
        self.dropout_prob = dropout_prob

        # Self Attention module for all decoder layers
        self.attns = nn.ModuleList(
            [
                nn.MultiheadAttention(d_model, num_heads,
                                      dropout=self.dropout_prob,
                                      batch_first=True)
                for _ in range(num_layers)
            ])

        # Cross Attention Module for all decoder layers
        self.cross_attns = nn.ModuleList(
            [
                nn.MultiheadAttention(d_model, num_heads,
                                      dropout=self.dropout_prob,
                                      batch_first=True)
                for _ in range(num_layers)
            ])

        # MLP Module for all decoder layers
        self.ffs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, ff_inner_dim),
                    nn.ReLU(),
                    nn.Dropout(self.dropout_prob),
                    nn.Linear(ff_inner_dim, d_model),
                )
                for _ in range(num_layers)
            ])

        # Norm for Self Attention Module for all decoder layers
        self.attn_norms = nn.ModuleList(
                [
                    nn.LayerNorm(d_model)
                    for _ in range(num_layers)
                ])

        # Norm for Cross Attention Module for all decoder layers
        self.cross_attn_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Norm for MLP Module for all decoder layers
        self.ff_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Dropout for Attention Module for all decoder layers
        self.attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Dropout for Cross Attention Module for all decoder layers
        self.cross_attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Dropout for MLP Module for all decoder layers
        self.ff_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Shared Output norm for all decoder outputs
        self.output_norm = nn.LayerNorm(d_model)

    def forward(self, query_objects, encoder_output,
                query_embedding, spatial_position_embedding):
        out = query_objects
        decoder_outputs = []
        decoder_cross_attn_weights = []
        for i in range(self.num_layers):
            # Norm, Self Attention, Dropout and Residual
            in_attn = self.attn_norms[i](out)
            q = in_attn + query_embedding
            k = in_attn + query_embedding
            out_attn, _ = self.attns[i](
                query=q,
                key=k,
                value=in_attn
            )
            out_attn = self.attn_dropouts[i](out_attn)
            out = out + out_attn

            # Norm, Cross Attention, Dropout and Residual
            in_attn = self.cross_attn_norms[i](out)
            q = in_attn + query_embedding
            k = encoder_output + spatial_position_embedding
            out_attn, decoder_cross_attn = self.cross_attns[i](
                query=q,
                key=k,
                value=encoder_output
            )
            decoder_cross_attn_weights.append(decoder_cross_attn)
            out_attn = self.cross_attn_dropouts[i](out_attn)
            out = out + out_attn

            # Norm, MLP, Dropout and Residual
            in_ff = self.ff_norms[i](out)
            out_ff = self.ffs[i](in_ff)
            out_ff = self.ff_dropouts[i](out_ff)
            out = out + out_ff
            decoder_outputs.append(self.output_norm(out))

        output = torch.stack(decoder_outputs)
        return output, torch.stack(decoder_cross_attn_weights)


In [18]:
class DETR(nn.Module):
    r"""
    DETR model class which instantiates all
    layers of DETR.
    A forward pass goes through the following layers:
        1. Backbone Call(currently frozen resnet 34)
        2. Backbone Featuremap Projection to d_model of transformer
        3. Encoder of Transformer
        4. Decoder of Transformer
        5. Class and BBox MLP
    """
    def __init__(self, config, num_classes, bg_class_idx, training = True):
        super().__init__()
        self.backbone_channels = config['backbone_channels']
        self.d_model = config['d_model']
        self.num_queries = config['num_queries']
        self.num_classes = num_classes
        self.num_decoder_layers = config['decoder_layers']
        self.cls_cost_weight = config['cls_cost_weight']
        self.l1_cost_weight = config['l1_cost_weight']
        self.giou_cost_weight = config['giou_cost_weight']
        self.bg_cls_weight = config['bg_class_weight']
        self.nms_threshold = config['nms_threshold']
        self.bg_class_idx = bg_class_idx
        self.training = training
        valid_bg_idx = (self.bg_class_idx == 0 or
                        self.bg_class_idx == (self.num_classes-1))
        assert valid_bg_idx, "Background can only be 0 or num_classes-1"

        self.backbone = nn.Sequential(*list(resnet34(
            weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1,
            norm_layer=torchvision.ops.FrozenBatchNorm2d
        ).children())[:-2])

        if config['freeze_backbone']:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.backbone_proj = nn.Conv2d(self.backbone_channels, self.d_model,
                                       kernel_size=1)
        self.encoder = TransformerEncoder(num_layers=config['encoder_layers'],
                                          num_heads=config['encoder_attn_heads'],
                                          d_model=config['d_model'],
                                          ff_inner_dim=config['ff_inner_dim'],
                                          dropout_prob=config['dropout_prob'])
        self.query_embed = nn.Parameter(torch.randn(self.num_queries, self.d_model))
        self.decoder = TransformerDecoder(num_layers=config['decoder_layers'],
                                          num_heads=config['decoder_attn_heads'],
                                          d_model=config['d_model'],
                                          ff_inner_dim=config['ff_inner_dim'],
                                          dropout_prob=config['dropout_prob'])
        self.class_mlp = nn.Linear(self.d_model, self.num_classes)
        self.bbox_mlp = nn.Sequential(
            nn.Linear(self.d_model, self.d_model),
            nn.ReLU(),
            nn.Linear(self.d_model, self.d_model),
            nn.ReLU(),
            nn.Linear(self.d_model, 4),
        )

    def forward(self, x, targets=None, score_thresh=0, use_nms=False):
        # x -> (B, C, H, W)
        # default d_model - 256
        # default C - 3
        # default H,W - 640,640
        # default feat_h,feat_w - 20,20

        conv_out = self.backbone(x)  # (B, C_back, feat_h, feat_w)
        # default C_back -  512

        conv_out = self.backbone_proj(conv_out)  # (B, d_model, feat_h, feat_w)

        batch_size, d_model, feat_h, feat_w = conv_out.shape
        spatial_pos_embed = get_spatial_position_embedding(self.d_model, conv_out)
        # spatial_pos_embed -> (feat_h * feat_w, d_model)

        conv_out = (conv_out.reshape(batch_size, d_model, feat_h * feat_w).
                    transpose(1, 2))
        # conv_out -> (B, feat_h*feat_w, d_model)

        # Encoder Call
        enc_output, enc_attn_weights = self.encoder(conv_out,  spatial_pos_embed)
        # enc_output -> (B, feat_h*feat_w, d_model)
        # enc_attn_weights -> (num_encoder_layers, B, feat_h*feat_w, feat_h*feat_w)

        query_objects = torch.zeros_like(self.query_embed.unsqueeze(0).
                                         repeat((batch_size, 1, 1)))
        # query_objects -> (B, num_queries, d_model)

        decoder_outputs = self.decoder(
            query_objects,
            enc_output,
            self.query_embed.unsqueeze(0).repeat((batch_size, 1, 1)),
            spatial_pos_embed)
        query_objects, decoder_attn_weights = decoder_outputs
        # query_objects -> (num_decoder_layers, B, num_queries, d_model)
        # decoder_attn_weights -> (num_decoder_layers, B, num_queries, feat_h*feat_w)

        cls_output = self.class_mlp(query_objects)
        # cls_output -> (num_decoder_layers, B, num_queries, num_classes)
        bbox_output = self.bbox_mlp(query_objects).sigmoid()
        # bbox_output -> (num_decoder_layers, B, num_queries, 4)

        losses = defaultdict(list)
        detections = []
        detr_output = {}

        if targets is not None:
            num_decoder_layers = self.num_decoder_layers
            # Perform matching for each decoder layer
            for decoder_idx in range(num_decoder_layers):
                cls_idx_output = cls_output[decoder_idx]
                bbox_idx_output = bbox_output[decoder_idx]
                with torch.no_grad():
                    # Concat all prediction boxes and class prob together
                    class_prob = cls_idx_output.reshape((-1, self.num_classes))
                    class_prob = class_prob.softmax(dim=-1)
                    # class_prob -> (B*num_queries, num_classes)

                    pred_boxes = bbox_idx_output.reshape((-1, 4))
                    # pred_boxes -> (B*num_queries, 4)

                    # Concat all target boxes and labels also together
                    target_labels = torch.cat([target["labels"] for target in targets])
                    target_boxes = torch.cat([target["boxes"] for target in targets])
                    # len(target_labels) -> num_targets_for_entire_batch
                    # target_boxes -> (num_targets_for_entire_batch, 4)

                    # Classification Cost
                    cost_classification = -class_prob[:, target_labels]
                    # cost_cls -> (B*num_queries, num_targets_for_entire_batch)

                    # DETR predicts cx,cy,w,h , we need to covert to x1y1x2y2 for giou
                    # Don't need to convert targets as they are already in x1y1x2y2
                    pred_boxes_x1y1x2y2 = torchvision.ops.box_convert(
                        pred_boxes,
                        'cxcywh',
                        'xyxy')

                    cost_localization_l1 = torch.cdist(
                        pred_boxes_x1y1x2y2,
                        target_boxes,
                        p=1
                     )
                    # cost_l1 -> (B*num_queries, num_targets_for_entire_batch)

                    cost_localization_giou = -torchvision.ops.generalized_box_iou(
                        pred_boxes_x1y1x2y2,
                        target_boxes
                    )
                    # cost_giou->(B*num_queries,num_targets_for_entire_batch)
                    total_cost = (self.l1_cost_weight * cost_localization_l1
                                  + self.cls_cost_weight * cost_classification
                                  + self.giou_cost_weight * cost_localization_giou)

                    total_cost = total_cost.reshape(batch_size,self.num_queries,-1).cpu()
                    # total_cost -> (B, num_queries, num_targets_for_entire_batch)

                    num_targets_per_image = [len(target["labels"]) for target in targets]
                    total_cost_per_batch_image = total_cost.split(
                        num_targets_per_image,
                        dim=-1
                    )
                    # total_cost_per_batch_image[0]=(B,num_queries,num_targets_0th_image)
                    # total_cost_per_batch_image[i]=(B,num_queries,num_targets_ith_image)

                    match_indices = []
                    for batch_idx in range(batch_size):
                        batch_idx_assignments = linear_sum_assignment(
                            total_cost_per_batch_image[batch_idx][batch_idx]
                        )
                        batch_idx_pred, batch_idx_target = batch_idx_assignments
                        # len(batch_idx_assignment_pred) = num_targets_ith_image
                        match_indices.append((torch.as_tensor(batch_idx_pred,
                                                              dtype=torch.int64),
                                              torch.as_tensor(batch_idx_target,
                                                              dtype=torch.int64)))
                        # match_indices -> [
                        #   ([pred_box_a1, ...],[target_box_i1, ...]),
                        #   ([pred_box_a2, ...],[target_box_i2, ...]),
                        #   ... assignment pairs for ith batch image
                        #   ]
                # pred_batch_idxs are batch indexes for each assignment pair
                pred_batch_idxs = torch.cat([
                    torch.ones_like(pred_idx) * i
                    for i, (pred_idx, _) in enumerate(match_indices)
                ])
                # pred_batch_idxs -> (num_targets_for_entire_batch, )
                # pred_query_idx are prediction box indexes(out of num_queries)
                # for each assignment pair
                pred_query_idx = torch.cat([pred_idx for (pred_idx, _) in match_indices])
                # pred_query_idx -> (num_targets_for_entire_batch, )

                # For all assigned prediction boxes, get the target label
                valid_obj_target_cls = torch.cat([
                    target["labels"][target_obj_idx]
                    for target, (_, target_obj_idx) in zip(targets, match_indices)
                ])
                # valid_obj_target_cls -> (num_targets_for_entire_batch, )

                # Initialize target class for all predicted boxes to be background class
                target_classes = torch.full(
                    cls_idx_output.shape[:2],
                    fill_value=self.bg_class_idx,
                    dtype=torch.int64,
                    device=cls_idx_output.device
                )
                # target_classes -> (B, num_queries)

                # For predicted boxes that were assigned to some target,
                # update their target label accordingly
                target_classes[(pred_batch_idxs, pred_query_idx)] = valid_obj_target_cls

                # To ensure background class is not disproportionately attended by model
                cls_weights = torch.ones(self.num_classes)
                cls_weights[self.bg_class_idx] = self.bg_cls_weight

                # Compute classification loss
                loss_cls = torch.nn.functional.cross_entropy(
                    cls_idx_output.reshape(-1, self.num_classes),
                    target_classes.reshape(-1),
                    cls_weights.to(cls_idx_output.device))

                # Get pred box coordinates for all matched pred boxes
                matched_pred_boxes = bbox_idx_output[pred_batch_idxs, pred_query_idx]
                # matched_pred_boxes -> (num_targets_for_entire_batch, 4)

                # Get target box coordinates for all matched target boxes
                target_boxes = torch.cat([
                    target['boxes'][target_obj_idx]
                    for target, (_, target_obj_idx) in zip(targets, match_indices)],
                    dim=0
                )
                # target_boxes -> (num_targets_for_entire_batch, 4)

                # Convert matched pred boxes to x1y1x2y2 format
                matched_pred_boxes_x1y1x2y2 = torchvision.ops.box_convert(
                    matched_pred_boxes,
                    'cxcywh',
                    'xyxy'
                )
                # Don't need to convert target boxes as they are in x1y1x2y2 format
                # Compute L1 Localization loss
                loss_bbox = torch.nn.functional.l1_loss(
                    matched_pred_boxes_x1y1x2y2,
                    target_boxes,
                    reduction='none')
                loss_bbox = loss_bbox.sum() / matched_pred_boxes.shape[0]

                # Compute GIoU loss
                loss_giou = torchvision.ops.generalized_box_iou_loss(
                    matched_pred_boxes_x1y1x2y2,
                    target_boxes
                )
                loss_giou = loss_giou.sum() / matched_pred_boxes.shape[0]

                losses['classification'].append(loss_cls * self.cls_cost_weight)
                losses['bbox_regression'].append(
                    loss_bbox * self.l1_cost_weight
                    + loss_giou * self.giou_cost_weight
                )
            detr_output['loss'] = losses
        if not self.training:
            # For inference we are only interested in last layer outputs
            cls_output = cls_output[-1]
            bbox_output = bbox_output[-1]
            # cls_output -> (B, num_queries, num_classes)
            # bbox_output -> (B, num_queries, 4)

            prob = torch.nn.functional.softmax(cls_output, -1)

            # Get all query boxes and their best fg class as label
            if self.bg_class_idx == 0:
                scores, labels = prob[..., 1:].max(-1)
                labels = labels+1
            else:
                scores, labels = prob[..., :-1].max(-1)

            # convert to x1y1x2y2 format
            boxes = torchvision.ops.box_convert(bbox_output,
                                                'cxcywh',
                                                'xyxy')

            for batch_idx in range(boxes.shape[0]):
                scores_idx = scores[batch_idx]
                labels_idx = labels[batch_idx]
                boxes_idx = boxes[batch_idx]

                # Low score filtering
                keep_idxs = scores_idx >= score_thresh
                scores_idx = scores_idx[keep_idxs]
                boxes_idx = boxes_idx[keep_idxs]
                labels_idx = labels_idx[keep_idxs]

                # NMS filtering
                if use_nms:
                    keep_idxs = torchvision.ops.batched_nms(
                        boxes_idx,
                        scores_idx,
                        labels_idx,
                        iou_threshold=self.nms_threshold)
                    scores_idx = scores_idx[keep_idxs]
                    boxes_idx = boxes_idx[keep_idxs]
                    labels_idx = labels_idx[keep_idxs]
                detections.append(
                    {
                        "boxes": boxes_idx,
                        "scores": scores_idx,
                        "labels": labels_idx
                        ,
                    }
                )

            detr_output['detections'] = detections
            detr_output['enc_attn'] = enc_attn_weights
            detr_output['dec_attn'] = decoder_attn_weights
        return detr_output

# MOR Creation

In [ ]:
# MoR reference code

class MoRExpertRouter(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # MoR uses a scalar routing score per token 
        self.router_weights = nn.Linear(embed_dim, 1)
        self.router_func = nn.Sigmoid() 

    def forward(self, x, topk):
        # 1. Compute scalar scores g^r for each token 
        # x shape: (batch, current_active_len, embed_dim)
        scores = self.router_func(self.router_weights(x)).squeeze(-1) # (batch, active_len)
        
        # 2. Select top-k tokens based on scores [cite: 579, 584]
        weights, rel_indices = torch.topk(scores, topk, dim=1)
        
        # 3. Sort indices to maintain original token position order
        sorted_indices, sort_idx = torch.sort(rel_indices, dim=1)
        sorted_weights = torch.gather(weights, 1, sort_idx)
        
        return sorted_weights, sorted_indices

class MoRRecursionBlock(nn.Module):
    def __init__(self, layers, router, embed_dim):
        super().__init__()
        self.layers = layers  # Shared stack of layers 
        self.router = router
        self.embed_dim = embed_dim

    def forward(self, x, active_indices, topk):
        """
        x: Full hidden states (batch, seq_len, embed_dim)
        active_indices: Indices of tokens eligible for this step (Hierarchical Filtering) 
        """
        # 1. Gather tokens eligible for this recursion
        active_x = torch.gather(x, 1, active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))
        
        # 2. Routing: Select which tokens from the active set continue
        weights, rel_indices = self.router(active_x, topk)
        
        # 3. Map relative indices back to absolute sequence positions
        abs_indices = torch.gather(active_indices, 1, rel_indices)
        
        # 4. Extract selected patches for computation
        # This focuses computation only on tokens still active
        selected_patches = torch.gather(active_x, 1, 
            rel_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))
        
        # 5. Pass through the SHARED layers (The recursion block)
        hidden_states = selected_patches
        for layer in self.layers:
            hidden_states = layer(hidden_states)
            
        # 6. Apply router weights and prepare update 
        weighted_updates = hidden_states * weights.unsqueeze(-1)
        
        # 7. RECOMBINE: Scatter updates back into the full sequence 
        # This preserves unselected tokens (the residual/skipped path) 
        x = x.clone() # Avoid in-place issues if needed
        x.scatter_add_(1, abs_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim), 
                       weighted_updates)
        
        # Return updated sequence and the new active set (Hierarchical Filtering) 
        return x, abs_indices

In [ ]:
class MoR_Encoder(nn.Module):
    r"""
    Encoder for transformer of DETR.
    This has sequence of encoder layers.
    Each layer has the following modules:
        1. LayerNorm for Self Attention
        2. Self Attention
        3. LayerNorm for MLP
        4. MLP
    """
    def __init__(self, num_blocks, num_recursions, num_heads, d_model, ff_inner_dim,
                 dropout_prob=0.0, if_middle_cycle = False):
        super().__init__()
        self.num_blocks = num_blocks
        self.num_recursions = num_recursions
        self.dropout_prob = dropout_prob
        self.embed_dim = d_model
        self.active_indices = None
        
        # if if_middle_cycle is True:
        #     self.starte


        # Expert Routers for each Recursion Block
        self.exp_routers = nn.ModuleList(
            [
                MoRExpertRouter(d_model)
                for _ in range(num_blocks)
            ]
        )

        # Self Attention Module for all encoder layers
        self.attns = nn.ModuleList(
                [
                    nn.MultiheadAttention(d_model, num_heads,
                                          dropout=self.dropout_prob,
                                          batch_first=True)
                    for _ in range(num_blocks)
                ]
            )

        # MLP Module for all encoder layers
        self.ffs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, ff_inner_dim),
                    nn.ReLU(),
                    nn.Dropout(self.dropout_prob),
                    nn.Linear(ff_inner_dim, d_model),
                )
                for _ in range(num_blocks)
            ])

        # Norm for Self Attention for all encoder layers
        self.attn_norms = nn.ModuleList(
                [
                    nn.LayerNorm(d_model)
                    for _ in range(num_blocks)
                ])

        # Norm for MLP for all encoder layers
        self.ff_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_blocks)
            ])

        # Dropout for Self Attention for all encoder layers
        self.attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_blocks)
            ])

        # Dropout for MLP for all encoder layers
        self.ff_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_blocks)
            ])

        # Norm for encoder output
        self.output_norm = nn.LayerNorm(d_model)

    def forward(self, x, spatial_position_embedding, active_indices):
        # attn_weights = []
        selected_patches = None
        weights = None
        out = x
        # filtered = False
        for i in range(self.num_blocks):
            """
            x: Full hidden states (batch, seq_len, embed_dim)
            active_indices: Indices of tokens eligible for this step (Hierarchical Filtering) 
            """
            topk = int(((self.num_blocks - i) / self.num_blocks) * out.size(1))
            # print(topk)
            if (i > 0):
                if (i > 1):
                    out = x
                # 1. Gather tokens eligible for this recursion
                print
                active_out = torch.gather(out, 1, active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))
                # active_out = torch.gather(out, 1,
                #                     active_indices.unsqueeze(0).unsqueeze(-1).expand(out.size(0), -1, self.embed_dim))
                # 2. Routing: Select which tokens from the active set continue 
                weights, rel_indices = self.exp_routers[i](active_out, topk)
                # print(weights.size(), "weights shape")
                
                # 3. Map relative indices back to absolute sequence positions
                # abs_indices = torch.gather(active_indices, 1, rel_indices)
                # print(active_indices.shape)
                # print(rel_indices)
                active_indices = torch.gather(active_indices, 1, rel_indices)
                
                # 4. Extract selected patches for computation
                # This focuses computation only on tokens still active
                selected_patches = torch.gather(active_out, 1, 
                    rel_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))

            # Running through the same layers more than once as per MOR
            for _ in range(self.num_recursions):
                # Norm, Self Attention, Dropout and Residual
                if selected_patches is not None:
                    in_attn = self.attn_norms[i](selected_patches)
                    # print(spatial_position_embedding.shape)
                    # print(active_indices.shape)
                    # temp_pos_embeds = torch.gather(spatial_position_embedding, 1, active_indices.unsqueeze(-1).expand(-1, -1, spatial_position_embedding.size(1)))
                    temp_pos_embeds = spatial_position_embedding[active_indices]
                    q = in_attn + temp_pos_embeds
                    k = in_attn + temp_pos_embeds
                    out_attn, attn_weight = self.attns[i](
                        query=q,
                        key=k,
                        value=in_attn
                    )
                    # attn_weights.append(attn_weight)
                    out_attn = self.attn_dropouts[i](out_attn)
                    selected_patches = selected_patches + out_attn
        
                    # Norm, MLP, Dropout and Residual
                    
                    in_ff = self.ff_norms[i](selected_patches)
                    out_ff = self.ffs[i](in_ff)
                    out_ff = self.ff_dropouts[i](out_ff)
                    
                    selected_patches = selected_patches + out_ff
                else:
                    in_attn = self.attn_norms[i](out)
                # Add spatial position embedding
                # to q,k for self attention
                    q = in_attn + spatial_position_embedding
                    k = in_attn + spatial_position_embedding
                    out_attn, attn_weight = self.attns[i](
                        query=q,
                        key=k,
                        value=in_attn
                    )
                    # attn_weights.append(attn_weight)
                    out_attn = self.attn_dropouts[i](out_attn)
                    out = out + out_attn
        
                    # Norm, MLP, Dropout and Residual
                    in_ff = self.ff_norms[i](out)
                    out_ff = self.ffs[i](in_ff)
                    out_ff = self.ff_dropouts[i](out_ff)
                    out = out + out_ff

            if weights is not None:
                # 6. Apply router weights and prepare update
                
                weighted_out = selected_patches * weights.unsqueeze(-1)
                
                # 7. RECOMBINE: Scatter updates back into the full sequence 
                # This preserves unselected tokens (the residual/skipped path) 
                x = x.clone() # Avoid in-place issues if needed
                # print(active_indices.shape)
                x.scatter_add_(1, active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim), 
                               weighted_out) # changed from abs_indices
                # print((active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim), 
                #                weighted_out).shape)

        # The above few lines are updating the initial input 'x' after every recursion block, so when you go to the next
        # recursion block, you must use out = x again.
            
        out = self.output_norm(x)
        return out
        # return out, torch.stack(attn_weights)


In [ ]:
class MoR_Decoder(nn.Module):
    r"""
        Decoder for transformer of DETR.
        This has sequence of decoder layers.
        Each layer has the following modules:
            1. LayerNorm for Self Attention
            2. Self Attention
            3. LayerNorm for Cross Attention on
                Encoder Outputs
            4. Cross Attention
            3. LayerNorm for MLP
            4. MLP
    """
    def __init__(self, num_blocks, num_recursions, num_heads, d_model, ff_inner_dim,
                 dropout_prob=0.0, if_middle_cycle = False):
        super().__init__()
        self.num_blocks = num_blocks
        self.num_recursions = num_recursions
        self.dropout_prob = dropout_prob
        self.embed_dim = d_model
        self.active_indices = None

        # Expert Routers for each Recursion Block
        self.exp_routers = nn.ModuleList(
            [
                MoRExpertRouter(d_model)
                for _ in range(num_blocks)
            ]
        )

        # Self Attention module for all decoder layers
        self.attns = nn.ModuleList(
            [
                nn.MultiheadAttention(d_model, num_heads,
                                      dropout=self.dropout_prob,
                                      batch_first=True)
                for _ in range(num_blocks)
            ])

        # Cross Attention Module for all decoder layers
        self.cross_attns = nn.ModuleList(
            [
                nn.MultiheadAttention(d_model, num_heads,
                                      dropout=self.dropout_prob,
                                      batch_first=True)
                for _ in range(num_blocks)
            ])

        # MLP Module for all decoder layers
        self.ffs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, ff_inner_dim),
                    nn.ReLU(),
                    nn.Dropout(self.dropout_prob),
                    nn.Linear(ff_inner_dim, d_model),
                )
                for _ in range(num_blocks)
            ])

        # Norm for Self Attention Module for all decoder layers
        self.attn_norms = nn.ModuleList(
                [
                    nn.LayerNorm(d_model)
                    for _ in range(num_blocks)
                ])

        # Norm for Cross Attention Module for all decoder layers
        self.cross_attn_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_blocks)
            ])

        # Norm for MLP Module for all decoder layers
        self.ff_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_blocks)
            ])

        # Dropout for Attention Module for all decoder layers
        self.attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_blocks)
            ])

        # Dropout for Cross Attention Module for all decoder layers
        self.cross_attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_blocks)
            ])

        # Dropout for MLP Module for all decoder layers
        self.ff_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_blocks)
            ])

        # Shared Output norm for all decoder outputs
        self.output_norm = nn.LayerNorm(d_model)

    # self.decoder(
    #         query_objects,
    #         enc_output,
    #         self.query_embed.unsqueeze(0).repeat((batch_size, 1, 1)),
    #         spatial_pos_embed, decoder_active_indices)
    
    def forward(self, query_objects, encoder_output,
                query_embedding, spatial_position_embedding, active_indices):
        
        decoder_outputs = []
        decoder_cross_attn_weights = []
        selected_patches = None
        weights = None
        out = query_objects
        for i in range(self.num_blocks):
            # print(i)
            """
            out: Full hidden states (batch, num_queries, embed_dim)
            active_indices: Indices of tokens eligible for this step (Hierarchical Filtering) 
            """
            topk = int(((self.num_blocks - i) / self.num_blocks) * out.size(1))
            # print(topk)
            if(i > 0):
                # if (i > 1):
                #     out = query_objects
                # 1. Gather tokens eligible for this recursion
                active_out = torch.gather(out, 1, active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))
                # print(active_indices.shape)
                # print(active_out.shape)

                # 2. Routing: Select which tokens from the active set continue
                weights, rel_indices = self.exp_routers[i](active_out, topk)
                
                # 3. Map relative indices back to absolute sequence positions
                # abs_indices = torch.gather(active_indices, 1, rel_indices)
                active_indices = torch.gather(active_indices, 1, rel_indices)
                
                # 4. Extract selected patches for computation
                # This focuses computation only on tokens still active
                selected_patches = torch.gather(active_out, 1, 
                    rel_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))

# Running through the same layers more than once as per MOR
            for _ in range(self.num_recursions):
                # Norm, Self Attention, Dropout and Residual
                if selected_patches is not None:
                    in_attn = self.attn_norms[i](selected_patches)
                    # print(query_embedding.size())
                    # temp_pos_embeds = query_embedding[active_indices]
                    temp_pos_embeds = torch.gather(query_embedding, 1, active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim))
                    # print(temp_pos_embeds.size())
                    q = in_attn + temp_pos_embeds
                    k = in_attn + temp_pos_embeds
                    out_attn, _ = self.attns[i](
                        query=q,
                        key=k,
                        value=in_attn
                    )
                    out_attn = self.attn_dropouts[i](out_attn)
                    #Selected patches, not out
                    selected_patches = selected_patches + out_attn
                    
                    # Norm, Cross Attention, Dropout and Residual
                    in_attn = self.cross_attn_norms[i](selected_patches)
                    q = in_attn + temp_pos_embeds
                    k = encoder_output + spatial_position_embedding
                    out_attn, decoder_cross_attn = self.cross_attns[i](
                        query=q,
                        key=k,
                        value=encoder_output
                    )
                    decoder_cross_attn_weights.append(decoder_cross_attn)
                    out_attn = self.cross_attn_dropouts[i](out_attn)
                    selected_patches = selected_patches + out_attn
        
                    # Norm, MLP, Dropout and Residual
                    in_ff = self.ff_norms[i](selected_patches)
                    out_ff = self.ffs[i](in_ff)
                    out_ff = self.ff_dropouts[i](out_ff)
                    selected_patches = selected_patches + out_ff


                    # 7. RECOMBINE: Scatter updates back into the full sequence 
                    # This preserves unselected tokens (the residual/skipped path) 
                    query_objects = query_objects.clone() # Avoid in-place issues if needed
                    query_objects.scatter_add_(1, active_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim), 
                                   selected_patches) # changed from
                    out = query_objects
                    # if i == 0:
                    decoder_outputs.append(self.output_norm(out))
                else:
                    in_attn = self.attn_norms[i](out)
                    q = in_attn + query_embedding
                    k = in_attn + query_embedding
                    out_attn, _ = self.attns[i](
                        query=q,
                        key=k,
                        value=in_attn
                    )
                    out_attn = self.attn_dropouts[i](out_attn)
                    out = out + out_attn
        
                    # Norm, Cross Attention, Dropout and Residual
                    in_attn = self.cross_attn_norms[i](out)
                    q = in_attn + query_embedding
                    k = encoder_output + spatial_position_embedding
                    out_attn, decoder_cross_attn = self.cross_attns[i](
                        query=q,
                        key=k,
                        value=encoder_output
                    )
                    decoder_cross_attn_weights.append(decoder_cross_attn)
                    out_attn = self.cross_attn_dropouts[i](out_attn)
                    out = out + out_attn
        
                    # Norm, MLP, Dropout and Residual
                    in_ff = self.ff_norms[i](out)
                    out_ff = self.ffs[i](in_ff)
                    out_ff = self.ff_dropouts[i](out_ff)
                    out = out + out_ff
                    # if i == 0:
                    decoder_outputs.append(self.output_norm(out))

                    
                # decoder_outputs.append(self.output_norm(query_objects))
            
        output = torch.stack(decoder_outputs)
        return output
        # return output, torch.stack(decoder_cross_attn_weights)

In [ ]:
class MoR_DETR(nn.Module):
    r"""
    DETR model class which instantiates all
    layers of DETR.
    A forward pass goes through the following layers:
        1. Backbone Call(currently frozen resnet 34)
        2. Backbone Featuremap Projection to d_model of transformer
        3. Encoder of Transformer
        4. Decoder of Transformer
        5. Class and BBox MLP
    """
    def __init__(self, config, num_classes, bg_class_idx, device, training = True):
        super().__init__()
        self.backbone_channels = config['backbone_channels']
        self.d_model = config['d_model']
        self.num_queries = config['num_queries']
        self.num_classes = num_classes
        self.num_recursions = config['decoder_num_recursions'] # repetitions per block
        self.num_blocks = config['decoder_num_blocks']
        self.num_decoder_layers = self.num_recursions * self.num_blocks
        self.cls_cost_weight = config['cls_cost_weight']
        self.l1_cost_weight = config['l1_cost_weight']
        self.giou_cost_weight = config['giou_cost_weight']
        self.bg_cls_weight = config['bg_class_weight']
        self.nms_threshold = config['nms_threshold']
        self.bg_class_idx = bg_class_idx
        self.training = training
        valid_bg_idx = (self.bg_class_idx == 0 or
                        self.bg_class_idx == (self.num_classes-1))
        assert valid_bg_idx, "Background can only be 0 or num_classes-1"

        self.backbone = nn.Sequential(*list(resnet34(
            weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1,
            norm_layer=torchvision.ops.FrozenBatchNorm2d
        ).children())[:-2])

        if config['freeze_backbone']:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.backbone_proj = nn.Conv2d(self.backbone_channels, self.d_model,
                                       kernel_size=1)
        self.encoder = MoR_Encoder(num_recursions=config['encoder_num_recursions'], # repetitions per block
                                          num_blocks=config['encoder_num_blocks'],
                                          num_heads=config['encoder_attn_heads'],
                                          d_model=config['d_model'],
                                          ff_inner_dim=config['ff_inner_dim'],
                                          dropout_prob=config['dropout_prob'])
        self.query_embed = nn.Parameter(torch.randn(self.num_queries, self.d_model))
        self.decoder = MoR_Decoder(num_recursions=config['decoder_num_recursions'],
                                          num_blocks=config['decoder_num_blocks'],
                                          num_heads=config['decoder_attn_heads'],
                                          d_model=config['d_model'],
                                          ff_inner_dim=config['ff_inner_dim'],
                                          dropout_prob=config['dropout_prob'])
        self.class_mlp = nn.Linear(self.d_model, self.num_classes)
        self.bbox_mlp = nn.Sequential(
            nn.Linear(self.d_model, self.d_model),
            nn.ReLU(),
            nn.Linear(self.d_model, self.d_model),
            nn.ReLU(),
            nn.Linear(self.d_model, 4),
        )

        self.device = device

    def forward(self, x, targets=None, score_thresh=0, use_nms=False):
        # x -> (B, C, H, W)
        # default d_model - 256
        # default C - 3
        # default H,W - 640,640
        # default feat_h,feat_w - 20,20

        conv_out = self.backbone(x)  # (B, C_back, feat_h, feat_w)
        # default C_back -  512

        conv_out = self.backbone_proj(conv_out)  # (B, d_model, feat_h, feat_w)

        batch_size, d_model, feat_h, feat_w = conv_out.shape
        spatial_pos_embed = get_spatial_position_embedding(self.d_model, conv_out)
        # spatial_pos_embed -> (feat_h * feat_w, d_model)

        conv_out = (conv_out.reshape(batch_size, d_model, feat_h * feat_w).
                    transpose(1, 2))
        # conv_out -> (B, feat_h*feat_w, d_model)

        # Encoder Call
        encoder_active_indices = torch.arange(feat_h*feat_w,
                                              dtype=torch.int64,
                                              device=self.device).unsqueeze(0).expand(conv_out.size(0), -1)
        # print(encoder_active_indices.shape)
        enc_output = self.encoder(conv_out,  spatial_pos_embed, encoder_active_indices)
        # enc_output, enc_attn_weights = self.encoder(conv_out,  spatial_pos_embed, encoder_active_indices)
        # enc_output -> (B, feat_h*feat_w, d_model)
        # enc_attn_weights -> (num_encoder_layers, B, feat_h*feat_w, feat_h*feat_w)

        query_objects = torch.zeros_like(self.query_embed.unsqueeze(0).
                                         repeat((batch_size, 1, 1)))
        # query_objects -> (B, num_queries, d_model)

        decoder_active_indices = torch.arange(self.num_queries,
                                              dtype=torch.int64,
                                              device=self.device).unsqueeze(0).expand(enc_output.size(0), -1)
        # print(decoder_active_indices.size())
        decoder_outputs = self.decoder(
            query_objects,
            enc_output,
            self.query_embed.unsqueeze(0).repeat((batch_size, 1, 1)),
            spatial_pos_embed, decoder_active_indices)
        query_objects = decoder_outputs
        # query_objects, decoder_attn_weights = decoder_outputs
        # query_objects -> (num_decoder_layers, B, num_queries, d_model)
        # decoder_attn_weights -> (num_decoder_layers, B, num_queries, feat_h*feat_w)

        cls_output = self.class_mlp(query_objects)
        # cls_output -> (num_decoder_layers, B, num_queries, num_classes)
        bbox_output = self.bbox_mlp(query_objects).sigmoid()
        # bbox_output -> (num_decoder_layers, B, num_queries, 4)

        losses = defaultdict(list)
        detections = []
        detr_output = {}

        if targets is not None:
            num_decoder_layers = self.num_decoder_layers
            # Perform matching for each decoder layer
            for decoder_idx in range(num_decoder_layers):
                cls_idx_output = cls_output[decoder_idx]
                bbox_idx_output = bbox_output[decoder_idx]
                with torch.no_grad():
                    # Concat all prediction boxes and class prob together
                    class_prob = cls_idx_output.reshape((-1, self.num_classes))
                    class_prob = class_prob.softmax(dim=-1)
                    # class_prob -> (B*num_queries, num_classes)

                    pred_boxes = bbox_idx_output.reshape((-1, 4))
                    # pred_boxes -> (B*num_queries, 4)

                    # Concat all target boxes and labels also together
                    target_labels = torch.cat([target["labels"] for target in targets])
                    target_boxes = torch.cat([target["boxes"] for target in targets])
                    # len(target_labels) -> num_targets_for_entire_batch
                    # target_boxes -> (num_targets_for_entire_batch, 4)

                    # Classification Cost
                    cost_classification = -class_prob[:, target_labels]
                    # cost_cls -> (B*num_queries, num_targets_for_entire_batch)

                    # DETR predicts cx,cy,w,h , we need to covert to x1y1x2y2 for giou
                    # Don't need to convert targets as they are already in x1y1x2y2
                    pred_boxes_x1y1x2y2 = torchvision.ops.box_convert(
                        pred_boxes,
                        'cxcywh',
                        'xyxy')

                    cost_localization_l1 = torch.cdist(
                        pred_boxes_x1y1x2y2,
                        target_boxes,
                        p=1
                     )
                    # cost_l1 -> (B*num_queries, num_targets_for_entire_batch)

                    cost_localization_giou = -torchvision.ops.generalized_box_iou(
                        pred_boxes_x1y1x2y2,
                        target_boxes
                    )
                    # cost_giou->(B*num_queries,num_targets_for_entire_batch)
                    total_cost = (self.l1_cost_weight * cost_localization_l1
                                  + self.cls_cost_weight * cost_classification
                                  + self.giou_cost_weight * cost_localization_giou)

                    total_cost = total_cost.reshape(batch_size,self.num_queries,-1).cpu()
                    # total_cost -> (B, num_queries, num_targets_for_entire_batch)

                    num_targets_per_image = [len(target["labels"]) for target in targets]
                    total_cost_per_batch_image = total_cost.split(
                        num_targets_per_image,
                        dim=-1
                    )
                    # total_cost_per_batch_image[0]=(B,num_queries,num_targets_0th_image)
                    # total_cost_per_batch_image[i]=(B,num_queries,num_targets_ith_image)

                    match_indices = []
                    for batch_idx in range(batch_size):
                        batch_idx_assignments = linear_sum_assignment(
                            total_cost_per_batch_image[batch_idx][batch_idx]
                        )
                        batch_idx_pred, batch_idx_target = batch_idx_assignments
                        # len(batch_idx_assignment_pred) = num_targets_ith_image
                        match_indices.append((torch.as_tensor(batch_idx_pred,
                                                              dtype=torch.int64),
                                              torch.as_tensor(batch_idx_target,
                                                              dtype=torch.int64)))
                        # match_indices -> [
                        #   ([pred_box_a1, ...],[target_box_i1, ...]),
                        #   ([pred_box_a2, ...],[target_box_i2, ...]),
                        #   ... assignment pairs for ith batch image
                        #   ]
                # pred_batch_idxs are batch indexes for each assignment pair
                pred_batch_idxs = torch.cat([
                    torch.ones_like(pred_idx) * i
                    for i, (pred_idx, _) in enumerate(match_indices)
                ])
                # pred_batch_idxs -> (num_targets_for_entire_batch, )
                # pred_query_idx are prediction box indexes(out of num_queries)
                # for each assignment pair
                pred_query_idx = torch.cat([pred_idx for (pred_idx, _) in match_indices])
                # pred_query_idx -> (num_targets_for_entire_batch, )

                # For all assigned prediction boxes, get the target label
                valid_obj_target_cls = torch.cat([
                    target["labels"][target_obj_idx]
                    for target, (_, target_obj_idx) in zip(targets, match_indices)
                ])
                # valid_obj_target_cls -> (num_targets_for_entire_batch, )

                # Initialize target class for all predicted boxes to be background class
                target_classes = torch.full(
                    cls_idx_output.shape[:2],
                    fill_value=self.bg_class_idx,
                    dtype=torch.int64,
                    device=cls_idx_output.device
                )
                # target_classes -> (B, num_queries)

                # For predicted boxes that were assigned to some target,
                # update their target label accordingly
                target_classes[(pred_batch_idxs, pred_query_idx)] = valid_obj_target_cls

                # To ensure background class is not disproportionately attended by model
                cls_weights = torch.ones(self.num_classes)
                cls_weights[self.bg_class_idx] = self.bg_cls_weight

                # Compute classification loss
                loss_cls = torch.nn.functional.cross_entropy(
                    cls_idx_output.reshape(-1, self.num_classes),
                    target_classes.reshape(-1),
                    cls_weights.to(cls_idx_output.device))

                # Get pred box coordinates for all matched pred boxes
                matched_pred_boxes = bbox_idx_output[pred_batch_idxs, pred_query_idx]
                # matched_pred_boxes -> (num_targets_for_entire_batch, 4)

                # Get target box coordinates for all matched target boxes
                target_boxes = torch.cat([
                    target['boxes'][target_obj_idx]
                    for target, (_, target_obj_idx) in zip(targets, match_indices)],
                    dim=0
                )
                # target_boxes -> (num_targets_for_entire_batch, 4)

                # Convert matched pred boxes to x1y1x2y2 format
                matched_pred_boxes_x1y1x2y2 = torchvision.ops.box_convert(
                    matched_pred_boxes,
                    'cxcywh',
                    'xyxy'
                )
                # Don't need to convert target boxes as they are in x1y1x2y2 format
                # Compute L1 Localization loss
                loss_bbox = torch.nn.functional.l1_loss(
                    matched_pred_boxes_x1y1x2y2,
                    target_boxes,
                    reduction='none')
                loss_bbox = loss_bbox.sum() / matched_pred_boxes.shape[0]

                # Compute GIoU loss
                loss_giou = torchvision.ops.generalized_box_iou_loss(
                    matched_pred_boxes_x1y1x2y2,
                    target_boxes
                )
                loss_giou = loss_giou.sum() / matched_pred_boxes.shape[0]

                losses['classification'].append(loss_cls * self.cls_cost_weight)
                losses['bbox_regression'].append(
                    loss_bbox * self.l1_cost_weight
                    + loss_giou * self.giou_cost_weight
                )
            detr_output['loss'] = losses
        if not self.training:
            # For inference we are only interested in last layer outputs
            cls_output = cls_output[-1]
            bbox_output = bbox_output[-1]
            # cls_output -> (B, num_queries, num_classes)
            # bbox_output -> (B, num_queries, 4)

            prob = torch.nn.functional.softmax(cls_output, -1)

            # Get all query boxes and their best fg class as label
            if self.bg_class_idx == 0:
                scores, labels = prob[..., 1:].max(-1)
                labels = labels+1
            else:
                scores, labels = prob[..., :-1].max(-1)

            # convert to x1y1x2y2 format
            boxes = torchvision.ops.box_convert(bbox_output,
                                                'cxcywh',
                                                'xyxy')

            for batch_idx in range(boxes.shape[0]):
                scores_idx = scores[batch_idx]
                labels_idx = labels[batch_idx]
                boxes_idx = boxes[batch_idx]

                # Low score filtering
                keep_idxs = scores_idx >= score_thresh
                scores_idx = scores_idx[keep_idxs]
                boxes_idx = boxes_idx[keep_idxs]
                labels_idx = labels_idx[keep_idxs]

                # NMS filtering
                if use_nms:
                    keep_idxs = torchvision.ops.batched_nms(
                        boxes_idx,
                        scores_idx,
                        labels_idx,
                        iou_threshold=self.nms_threshold)
                    scores_idx = scores_idx[keep_idxs]
                    boxes_idx = boxes_idx[keep_idxs]
                    labels_idx = labels_idx[keep_idxs]
                detections.append(
                    {
                        "boxes": boxes_idx,
                        "scores": scores_idx,
                        "labels": labels_idx,
                    }
                )

            detr_output['detections'] = detections
            # detr_output['enc_attn'] = enc_attn_weights
            # detr_output['dec_attn'] = decoder_attn_weights
        return detr_output

# Training Pipeline

In [20]:
# DETR PIPELINE

# def train_detr(config_path):
#     # Read the config file #
#     with open(config_path, 'r') as file:
#         try:
#             config = yaml.safe_load(file)
#         except yaml.YAMLError as exc:
#             print(exc)
#     print(config)
#     #########################

#     dataset_config = config['dataset_params']
#     train_config = config['train_params']
#     model_config = config['model_params']

#     seed = train_config['seed']
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     random.seed(seed)

#     # voc = VOCDataset('train',
#     #                  im_sets=dataset_config['train_im_sets'],
#     #                  im_size=dataset_config['im_size'])
#     # train_dataset = DataLoader(voc,
#     #                            batch_size=train_config['batch_size'],
#     #                            shuffle=True,
#     #                            collate_fn=collate_function)

#     # Instantiate model and load checkpoint if present
#     model = DETR(
#         config=model_config,
#         num_classes=dataset_config['num_classes'],
#         bg_class_idx=dataset_config['bg_class_idx']
#     )
#     model.to(device)
#     model.train()

#     if os.path.exists(os.path.join(train_config['task_name'],
#                                    train_config['ckpt_name'])):

#         state_dict = torch.load(
#             os.path.join(train_config['task_name'],
#                          train_config['ckpt_name']),
#             map_location=device)
#         model.load_state_dict(state_dict)
#         print('Loading checkpoint as one exists')

#     if not os.path.exists(train_config['task_name']):
#         os.mkdir(train_config['task_name'])

#     optimizer = torch.optim.AdamW(lr=train_config['lr'],
#                                   params=filter(lambda p: p.requires_grad,
#                                                 model.parameters()),
#                                   weight_decay=1E-4)

#     # backbone_params = [
#     #     p for n, p in model.named_parameters() if 'backbone.' in n]
#     # transformer_params = [
#     #     p for n, p in model.named_parameters() if 'backbone.' not in n]
#     # optimizer = torch.optim.AdamW([
#     #     {'params': backbone_params, 'lr': train_config['lr']*0.1},
#     #     {'params': transformer_params, 'lr': train_config['lr']},
#     # ], weight_decay=1e-4)

#     lr_scheduler = MultiStepLR(optimizer,
#                                milestones=train_config['lr_steps'],
#                                gamma=0.1)
#     acc_steps = train_config['acc_steps']
#     num_epochs = train_config['num_epochs']
#     steps = 0
#     for i in range(num_epochs):
#         detr_classification_losses = []
#         detr_localization_losses = []
#         for idx, (ims, targets, _) in enumerate(tqdm(train_dataset)):
#             for target in targets:
#                 target['boxes'] = target['boxes'].float().to(device)
#                 target['labels'] = target['labels'].long().to(device)
#             images = torch.stack([im.float().to(device) for im in ims], dim=0)
#             batch_losses = model(images, targets)['loss']

#             loss = (sum(batch_losses['classification']) +
#                     sum(batch_losses['bbox_regression']))

#             detr_classification_losses.append(sum(batch_losses['classification']).item())
#             detr_localization_losses.append(sum(batch_losses['bbox_regression']).item())
#             loss = loss / acc_steps
#             loss.backward()

#             if (idx + 1) % acc_steps == 0:
#                 optimizer.step()
#                 optimizer.zero_grad()
#             if steps % train_config['log_steps'] == 0:
#                 loss_output = ''
#                 loss_output += 'DETR Classification Loss : {:.4f}'.format(
#                     np.mean(detr_classification_losses))
#                 loss_output += ' | DETR Localization Loss : {:.4f}'.format(
#                     np.mean(detr_localization_losses))
#                 print(loss_output, lr_scheduler.get_last_lr())
#             if torch.isnan(loss):
#                 print('Loss is becoming nan. Exiting')
#                 exit(0)
#             steps += 1
#         optimizer.step()
#         optimizer.zero_grad()
#         lr_scheduler.step()
#         print('Finished epoch {}'.format(i+1))
#         loss_output = ''
#         loss_output += 'DETR Classification Loss : {:.4f}'.format(
#             np.mean(detr_classification_losses))
#         loss_output += ' | DETR Localization Loss : {:.4f}'.format(
#             np.mean(detr_localization_losses))
#         print(loss_output)
#         torch.save(model.state_dict(), os.path.join(train_config['task_name'],
#                                                          train_config['ckpt_name']))
#     print('Done Training...')

In [21]:
def save_checkpoint(model, optimizer, scheduler, epoch, loss, steps):
    
    with open(config_path_mor, 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    #     print(config)
    train_config = config['train_params']

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
        'steps' : steps,
    }
    torch.save(checkpoint, os.path.join(train_config['task_name'],
                                                         train_config['ckpt_name']))
    print(f"Checkpoint saved at epoch {epoch}")

In [22]:
# result_path = "train_results.txt"

# with open(result_path, 'a') as log:
#     log_entry = "I love AI/ML"
#     log.write(log_entry)

In [26]:
# MOR PIPELINE

# result_path = "train_mor_results.txt"
result_path = "train_mor_results.txt"

def train_mor_detr(config_mor_path):
    # Read the config file #
    with open(config_mor_path, 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    print(config)
    #########################

    dataset_config = config['dataset_params']
    train_config = config['train_params']
    model_config = config['model_params']

    seed = train_config['seed']
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # voc = VOCDataset('train',
    #                  im_sets=dataset_config['train_im_sets'],
    #                  im_size=dataset_config['im_size'])
    # train_dataset = DataLoader(voc,
    #                            batch_size=train_config['batch_size'],
    #                            shuffle=True,
    #                            collate_fn=collate_function)

    # Instantiate model and load checkpoint if present
    model = MoR_DETR(
        config=model_config,
        num_classes=dataset_config['num_classes'],
        bg_class_idx=dataset_config['bg_class_idx'],
        device=device
    )
    model.to(device)

    optimizer = torch.optim.AdamW(lr=train_config['lr'],
                              params=filter(lambda p: p.requires_grad,
                                            model.parameters()),
                              weight_decay=1E-4)
    
    lr_scheduler = MultiStepLR(optimizer,
                           milestones=train_config['lr_steps'],
                           gamma=0.1)

    epoch = 0
    last_loss = 0
    steps = 0
    if os.path.exists(os.path.join(train_config['task_name'],
                                   train_config['ckpt_name'])):

        state_dict = torch.load(
            os.path.join(train_config['task_name'],
                         train_config['ckpt_name']),
            map_location=device)
        model.load_state_dict(state_dict["model_state_dict"])
        optimizer.load_state_dict(state_dict['optimizer_state_dict'])
        lr_scheduler.load_state_dict(state_dict['scheduler_state_dict'])
        epoch = (state_dict['epoch'])
        steps = state_dict['steps']
        print('Loading checkpoint as one exists')

    if not os.path.exists(train_config['task_name']):
        os.mkdir(train_config['task_name'])

    optimizer = torch.optim.AdamW(lr=train_config['lr'],
                                  params=filter(lambda p: p.requires_grad,
                                                model.parameters()),
                                  weight_decay=1E-4)

    # backbone_params = [
    #     p for n, p in model.named_parameters() if 'backbone.' in n]
    # transformer_params = [
    #     p for n, p in model.named_parameters() if 'backbone.' not in n]
    # optimizer = torch.optim.AdamW([
    #     {'params': backbone_params, 'lr': train_config['lr']*0.1},
    #     {'params': transformer_params, 'lr': train_config['lr']},
    # ], weight_decay=1e-4)

    acc_steps = train_config['acc_steps']
    num_epochs = train_config['num_epochs']

    for i in range(epoch, num_epochs):
        model.train()
        last_epoch = i
        mor_classification_losses = []
        mor_localization_losses = []
        for idx, (ims, targets, _) in enumerate(tqdm(train_dataset)):
            for target in targets:
                target['boxes'] = target['boxes'].float().to(device)
                target['labels'] = target['labels'].long().to(device)
            images = torch.stack([im.float().to(device) for im in ims], dim=0)
            batch_losses = model(images, targets)['loss']

            loss = (sum(batch_losses['classification']) +
                    sum(batch_losses['bbox_regression']))

            mor_classification_losses.append(sum(batch_losses['classification']).item())
            mor_localization_losses.append(sum(batch_losses['bbox_regression']).item())
            loss = loss / acc_steps
            last_loss = loss
            loss.backward()

            if (idx + 1) % acc_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
            if steps % train_config['log_steps'] == 0:
                loss_output = ''
                loss_output += 'MOR Classification Loss : {:.4f}'.format(
                    np.mean(mor_classification_losses))
                loss_output += ' | MOR Localization Loss : {:.4f}'.format(
                    np.mean(mor_localization_losses))
                print(loss_output, lr_scheduler.get_last_lr())
            if torch.isnan(loss):
                with open(result_path, 'a') as log:
                    log_entry = 'Loss is becoming nan. Exiting'
                    log.write(log_entry)
                exit(0)
            steps += 1
            last_steps = steps
            
        optimizer.step()
        optimizer.zero_grad()
        # --- VALIDATION PHASE ---
        model.eval()
        val_classification_losses = []
        val_localization_losses = []
        
        print(f"Running Validation for Epoch {i+1}...")
        with torch.no_grad():
            for ims, targets, _ in tqdm(val_dataset, desc="Validating"):
                for target in targets:
                    target['boxes'] = target['boxes'].float().to(device)
                    target['labels'] = target['labels'].long().to(device)
                images = torch.stack([im.float().to(device) for im in ims], dim=0)
                
                batch_losses = model(images, targets)['loss']
                
                val_classification_losses.append(sum(batch_losses['classification']).item())
                val_localization_losses.append(sum(batch_losses['bbox_regression']).item())
        
        lr_scheduler.step()
        print('Finished epoch {}'.format(i+1))
        loss_output = ''
        loss_output += 'MOR Classification Loss : {:.4f}'.format(
            np.mean(mor_classification_losses))
        loss_output += ' | MOR Localization Loss : {:.4f}'.format(
            np.mean(mor_localization_losses))
        print(loss_output)

# Log Validation Results
        print('Val Class Loss: {:.4f} | Val Loc Loss: {:.4f}'.format(
            np.mean(val_classification_losses), np.mean(val_localization_losses)))
        
        # torch.save(model.state_dict(), os.path.join(train_config['task_name'],
        #                                                  train_config['ckpt_name']))
        
        with open(result_path, 'a') as log:
            log_entry = (f"Epoch {i+1}: "
                 f"Train_Class_Loss={np.mean(mor_classification_losses):.6f}, "
                 f"Train_Loc_Loss={np.mean(mor_localization_losses):.6f}, "
                 f"Val_Class_Loss={np.mean(val_classification_losses):.6f}, "
                 f"Val_Loc_Loss={np.mean(val_localization_losses):.6f}\n")
            log.write(log_entry)
        save_checkpoint(model, optimizer, lr_scheduler, last_epoch, last_loss, last_steps)
    with open(result_path, 'a') as log:
        log_entry = 'Done Training...'
        log.write(log_entry)
    # print('Done Training...')

In [ ]:
# DO IT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_mor_detr(config_path_mor)

{'dataset_params': {'train_im_sets': ['/scratch/abhilashk/Pascal-VOC/VOCdevkit/VOC2007', '/scratch/abhilashk/Pascal-VOC/VOCdevkit/VOC2012'], 'test_im_sets': ['/scratch/abhilashk/Pascal-VOC/VOCdevkit/VOC2007'], 'num_classes': 21, 'bg_class_idx': 0, 'im_size': 640}, 'model_params': {'im_channels': 3, 'backbone_channels': 512, 'd_model': 256, 'num_queries': 25, 'freeze_backbone': True, 'encoder_num_recursions': 2, 'encoder_num_blocks': 4, 'encoder_attn_heads': 8, 'decoder_num_recursions': 2, 'decoder_num_blocks': 4, 'decoder_attn_heads': 8, 'dropout_prob': 0.1, 'ff_inner_dim': 2048, 'cls_cost_weight': 1.0, 'l1_cost_weight': 5.0, 'giou_cost_weight': 2.0, 'bg_class_weight': 0.1, 'nms_threshold': 0.5}, 'train_params': {'task_name': 'mor_voc', 'eval_score_threshold': 0.0, 'infer_score_threshold': 0.5, 'use_nms_eval': False, 'use_nms_infer': True, 'seed': 1111, 'acc_steps': 1, 'num_epochs': 75, 'batch_size': 4, 'lr_steps': [200], 'lr': 0.0001, 'log_steps': 500, 'ckpt_name': 'L-mor_detr_voc2007

  0%|          | 0/3307 [00:00<?, ?it/s]/home/abhilashk/.conda/envs/MoR_Test/lib/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
  0%|          | 1/3307 [00:36<33:18:10, 36.26s/it]

MOR Classification Loss : 26.8421 | MOR Localization Loss : 53.9779 [0.0001]


 13%|█▎        | 425/3307 [04:04<23:15,  2.06it/s] 

# Testing Pipeline

In [ ]:
def get_iou(det, gt):
    det_x1, det_y1, det_x2, det_y2 = det
    gt_x1, gt_y1, gt_x2, gt_y2 = gt

    x_left = max(det_x1, gt_x1)
    y_top = max(det_y1, gt_y1)
    x_right = min(det_x2, gt_x2)
    y_bottom = min(det_y2, gt_y2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    area_intersection = (x_right - x_left) * (y_bottom - y_top)
    det_area = (det_x2 - det_x1) * (det_y2 - det_y1)
    gt_area = (gt_x2 - gt_x1) * (gt_y2 - gt_y1)
    area_union = float(det_area + gt_area - area_intersection + 1E-6)
    iou = area_intersection / area_union
    return iou


def compute_map(det_boxes, gt_boxes, iou_threshold=0.5, method='area', difficult=None):
    # det_boxes = [
    #   {
    #       'person' : [[x1, y1, x2, y2, score], ...],
    #       'car' : [[x1, y1, x2, y2, score], ...]
    #   }
    #   {det_boxes_img_2},
    #   ...
    #   {det_boxes_img_N},
    # ]
    #
    # gt_boxes = [
    #   {
    #       'person' : [[x1, y1, x2, y2], ...],
    #       'car' : [[x1, y1, x2, y2], ...]
    #   },
    #   {gt_boxes_img_2},
    #   ...
    #   {gt_boxes_img_N},
    # ]

    gt_labels = {cls_key for im_gt in gt_boxes for cls_key in im_gt.keys()}
    gt_labels = sorted(gt_labels)

    all_aps = {}
    # average precisions for ALL classes
    aps = []
    for idx, label in enumerate(gt_labels):
        # Get detection predictions of this class
        cls_dets = [
            [im_idx, im_dets_label] for im_idx, im_dets in enumerate(det_boxes)
            if label in im_dets for im_dets_label in im_dets[label]
        ]

        # cls_dets = [
        #   (0, [x1_0, y1_0, x2_0, y2_0, score_0]),
        #   ...
        #   (0, [x1_M, y1_M, x2_M, y2_M, score_M]),
        #   (1, [x1_0, y1_0, x2_0, y2_0, score_0]),
        #   ...
        #   (1, [x1_N, y1_N, x2_N, y2_N, score_N]),
        #   ...
        # ]

        # Sort them by confidence score
        cls_dets = sorted(cls_dets, key=lambda k: -k[1][-1])

        # For tracking which gt boxes of this class have already been matched
        gt_matched = [[False for _ in im_gts[label]] for im_gts in gt_boxes]
        # Number of gt boxes for this class for recall calculation
        num_gts = sum([len(im_gts[label]) for im_gts in gt_boxes])
        num_difficults = sum([sum(difficults_label[label])
                              for difficults_label in difficult])

        tp = [0] * len(cls_dets)
        fp = [0] * len(cls_dets)

        # For each prediction
        for det_idx, (im_idx, det_pred) in enumerate(cls_dets):
            # Get gt boxes for this image and this label
            im_gts = gt_boxes[im_idx][label]
            im_gt_difficults = difficult[im_idx][label]

            max_iou_found = -1
            max_iou_gt_idx = -1

            # Get best matching gt box
            for gt_box_idx, gt_box in enumerate(im_gts):
                gt_box_iou = get_iou(det_pred[:-1], gt_box)
                if gt_box_iou > max_iou_found:
                    max_iou_found = gt_box_iou
                    max_iou_gt_idx = gt_box_idx
            # TP only if iou >= threshold and this gt has not yet been matched
            if max_iou_found >= iou_threshold:
                if not gt_matched[im_idx][max_iou_gt_idx]:
                    # If tp then we set this gt box as matched
                    gt_matched[im_idx][max_iou_gt_idx] = True
                    tp[det_idx] = 1
                else:
                    fp[det_idx] = 1
            else:
                fp[det_idx] = 1

        # Cumulative tp and fp
        tp = np.cumsum(tp)
        fp = np.cumsum(fp)

        eps = np.finfo(np.float32).eps
        # recalls = tp / np.maximum(num_gts, eps)
        recalls = tp / np.maximum(num_gts - num_difficults, eps)
        precisions = tp / np.maximum((tp + fp), eps)

        if method == 'area':
            recalls = np.concatenate(([0.0], recalls, [1.0]))
            precisions = np.concatenate(([0.0], precisions, [0.0]))

            # Replace precision values with recall r with maximum precision value
            # of any recall value >= r
            # This computes the precision envelope
            for i in range(precisions.size - 1, 0, -1):
                precisions[i - 1] = np.maximum(precisions[i - 1], precisions[i])
            # For computing area, get points where recall changes value
            i = np.where(recalls[1:] != recalls[:-1])[0]
            # Add the rectangular areas to get ap
            ap = np.sum((recalls[i + 1] - recalls[i]) * precisions[i + 1])
        elif method == 'interp':
            ap = 0.0
            for interp_pt in np.arange(0, 1 + 1E-3, 0.1):
                # Get precision values for recall values >= interp_pt
                prec_interp_pt = precisions[recalls >= interp_pt]

                # Get max of those precision values
                prec_interp_pt= prec_interp_pt.max() if prec_interp_pt.size>0.0 else 0.0
                ap += prec_interp_pt
            ap = ap / 11.0
        else:
            raise ValueError('Method can only be area or interp')
        if num_gts > 0:
            aps.append(ap)
            all_aps[label] = ap
        else:
            all_aps[label] = np.nan
    # compute mAP at provided iou threshold
    mean_ap = sum(aps) / len(aps)
    return mean_ap, all_aps


def load_model_and_dataset(args):
    # Read the config file #
    with open(args.config_path, 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    print(config)
    ########################

    dataset_config = config['dataset_params']
    model_config = config['model_params']
    train_config = config['train_params']

    voc = VOCDataset('test',
                     im_sets=dataset_config['test_im_sets'],
                     im_size=dataset_config['im_size'])
    test_dataset = DataLoader(voc, batch_size=1, shuffle=False)

    model = DETR(
        config=model_config,
        num_classes=dataset_config['num_classes'],
        bg_class_idx=dataset_config['bg_class_idx']
    )
    model.to(device=torch.device(device))
    model.eval()

    assert os.path.exists(os.path.join(train_config['task_name'],
                                       train_config['ckpt_name'])), \
        "No checkpoint exists at {}".format(os.path.join(train_config['task_name'],
                                                         train_config['ckpt_name']))
    model.load_state_dict(torch.load(os.path.join(train_config['task_name'],
                                                       train_config['ckpt_name']),
                                     map_location=device))
    return model, voc, test_dataset, config


def infer(args):
    if not os.path.exists('samples'):
        os.mkdir('samples')

    model, voc, test_dataset, config = load_model_and_dataset(args)
    import cv2
    num_samples = 5
    for i in tqdm(range(num_samples)):
        dataset_idx = random.randint(0, len(voc))
        im_tensor, target, fname = voc[dataset_idx]
        detr_output = model(
            im_tensor.unsqueeze(0).to(device),
            score_thresh=config['train_params']['infer_score_threshold'],
            use_nms=config['train_params']['use_nms_infer']
        )
        detr_detections = detr_output['detections']
        enc_attn_weights = detr_output['enc_attn']
        dec_attn_weights = detr_output['dec_attn']

        gt_im = cv2.imread(fname)
        h, w = gt_im.shape[:2]
        gt_im_copy = gt_im.copy()
        # Saving images with ground truth boxes
        for idx, box in enumerate(target['boxes']):
            x1, y1, x2, y2 = box.detach().cpu().numpy()
            x1, y1, x2, y2 = int(w*x1), int(h*y1), int(w*x2), int(h*y2)
            cv2.rectangle(gt_im, (x1, y1), (x2, y2), thickness=2, color=[0, 255, 0])
            cv2.rectangle(gt_im_copy, (x1, y1), (x2, y2), thickness=2, color=[0, 255, 0])
            text = voc.idx2label[target['labels'][idx].detach().cpu().item()]
            text_size, _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_PLAIN, 1, 1)
            text_w, text_h = text_size
            cv2.rectangle(gt_im_copy, (x1, y1), (x1 + 10 + text_w, y1 + 10 + text_h), [255, 255, 255], -1)
            cv2.putText(gt_im, text=voc.idx2label[target['labels'][idx].detach().cpu().item()],
                        org=(x1 + 5, y1 + 15),
                        thickness=1,
                        fontScale=1,
                        color=[0, 0, 0],
                        fontFace=cv2.FONT_HERSHEY_PLAIN)
            cv2.putText(gt_im_copy, text=text,
                        org=(x1 + 5, y1 + 15),
                        thickness=1,
                        fontScale=1,
                        color=[0, 0, 0],
                        fontFace=cv2.FONT_HERSHEY_PLAIN)
        cv2.addWeighted(gt_im_copy, 0.7, gt_im, 0.3, 0, gt_im)
        cv2.imwrite('samples/output_detr_gt_{}.png'.format(i), gt_im)

        # Getting predictions from trained model
        boxes = detr_detections[0]['boxes']
        labels = detr_detections[0]['labels']
        scores = detr_detections[0]['scores']
        im = cv2.imread(fname)
        im_copy = im.copy()

        # Saving images with predicted boxes
        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = box.detach().cpu().numpy()
            x1, y1, x2, y2 = int(w*x1), int(h*y1), int(w*x2), int(h*y2)
            cv2.rectangle(im, (x1, y1), (x2, y2), thickness=2, color=[0, 0, 255])
            cv2.rectangle(im_copy, (x1, y1), (x2, y2), thickness=2, color=[0, 0, 255])
            text = '{} : {:.2f}'.format(voc.idx2label[labels[idx].detach().cpu().item()],
                                        scores[idx].detach().cpu().item())
            text_size, _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_PLAIN, 1, 1)
            text_w, text_h = text_size
            cv2.rectangle(im_copy, (x1, y1), (x1 + 10 + text_w, y1 + 10 + text_h), [255, 255, 255], -1)
            cv2.putText(im, text=text,
                        org=(x1 + 5, y1 + 15),
                        thickness=1,
                        fontScale=1,
                        color=[0, 0, 0],
                        fontFace=cv2.FONT_HERSHEY_PLAIN)
            cv2.putText(im_copy, text=text,
                        org=(x1 + 5, y1 + 15),
                        thickness=1,
                        fontScale=1,
                        color=[0, 0, 0],
                        fontFace=cv2.FONT_HERSHEY_PLAIN)
        cv2.addWeighted(im_copy, 0.7, im, 0.3, 0, im)
        cv2.imwrite('samples/output_detr_{}.jpg'.format(i), im)
    print('Done Detecting...')


def evaluate_map(args):
    model, voc, test_dataset, config = load_model_and_dataset(args)

    gts = []
    preds = []
    difficults = []
    for im_tensor, target, fname in tqdm(test_dataset):
        im_tensor = im_tensor.float().to(device)
        target_bboxes = target['boxes'].float()[0].to(device)
        target_labels = target['labels'].long()[0].to(device)
        difficult = target['difficult'].long()[0].to(device)
        detr_output = model(
            im_tensor,
            score_thresh=config['train_params']['eval_score_threshold'],
            use_nms=config['train_params']['use_nms_eval']
        )
        detr_detections = detr_output['detections']

        boxes = detr_detections[0]['boxes']
        labels = detr_detections[0]['labels']
        scores = detr_detections[0]['scores']

        pred_boxes = {}
        gt_boxes = {}
        difficult_boxes = {}

        for label_name in voc.label2idx:
            pred_boxes[label_name] = []
            gt_boxes[label_name] = []
            difficult_boxes[label_name] = []

        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = box.detach().cpu().numpy()
            label = labels[idx].detach().cpu().item()
            score = scores[idx].detach().cpu().item()
            label_name = voc.idx2label[label]
            pred_boxes[label_name].append([x1, y1, x2, y2, score])
        for idx, box in enumerate(target_bboxes):
            x1, y1, x2, y2 = box.detach().cpu().numpy()
            label = target_labels[idx].detach().cpu().item()
            label_name = voc.idx2label[label]
            gt_boxes[label_name].append([x1, y1, x2, y2])
            difficult_boxes[label_name].append(difficult[idx].detach().cpu().item())

        gts.append(gt_boxes)
        preds.append(pred_boxes)
        difficults.append(difficult_boxes)

    mean_ap, all_aps = compute_map(preds, gts, method='area', difficult=difficults)
    print('Class Wise Average Precisions')
    for idx in range(len(voc.idx2label)):
        print('AP for class {} = {:.4f}'.format(voc.idx2label[idx],
                                                all_aps[voc.idx2label[idx]]))
    print('Mean Average Precision : {:.4f}'.format(mean_ap))